In [ ]:
from google.colab import files

uploaded = files.upload()



In [ ]:
import pandas as pd

df = pd.read_csv("main.csv", encoding="ISO-8859-1")  # Alternative: encoding="latin-1"
print(df.head())  # Show first few rows


In [ ]:
# Display first few rows
print(df.head())

# Check column names
print(df.columns)

# Check for missing values
print(df.isnull().sum())


In [ ]:
!pip install transformers datasets torch scikit-learn


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# Load tokenizer (RoBERTa is good for classification)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True)

# Apply tokenization
dataset = dataset.map(tokenize_function, batched=True)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Convert "Fake" and "Real" to numerical labels
df["Label"] = df["Label"].map({"Fake": 0, "Real": 1})

# Compute class weights
class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=df["Label"])
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

print("Class Weights:", class_weights_dict)


In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into train (80%) and test (20%)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["title"], df["Label"], test_size=0.2, stratify=df["Label"], random_state=42
)

print("Training set size:", len(train_texts))
print("Testing set size:", len(test_texts))


In [ ]:
# Tokenize train and test sets
train_encodings = tokenizer(list(train_texts), padding=True, truncation=True, max_length=512)
test_encodings = tokenizer(list(test_texts), padding=True, truncation=True, max_length=512)

# Convert labels to lists
train_labels = list(train_labels)
test_labels = list(test_labels)


In [ ]:
import torch

# Define a PyTorch dataset
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# Create dataset objects
train_dataset = FakeNewsDataset(train_encodings, train_labels)
test_dataset = FakeNewsDataset(test_encodings, test_labels)


In [ ]:
from transformers import AutoModelForSequenceClassification

# Load RoBERTa model with 2 output labels (Fake = 0, Real = 1)
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW


# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define class weights for Fake (0) and Real (1)
class_weights_tensor = torch.tensor([class_weights_dict[0], class_weights_dict[1]]).to(device)

# Use weighted loss function (CrossEntropyLoss)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

# Define optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)


In [ ]:
# Define batch size
batch_size = 8

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
from tqdm import tqdm

# Number of training epochs
epochs = 4

# Training loop
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()  # Clear previous gradients

        # Move batch to GPU if available
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # Compute loss
        total_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Average Training Loss: {avg_loss:.4f}\n")


Hyper parameter

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Set model to evaluation mode
model.eval()

# Store predictions & true labels
predictions = []
true_labels = []

# Disable gradient calculation for evaluation (faster & memory-efficient)
with torch.no_grad():
    for batch in tqdm(test_loader):
        # Move batch to GPU if available
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Get model predictions
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)  # Convert logits to class labels

        # Store results
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Compute accuracy, precision, recall, and F1-score
accuracy = accuracy_score(true_labels, predictions)
report = classification_report(true_labels, predictions, target_names=["Fake", "Real"])

print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:\n", report)
